<a href="https://colab.research.google.com/github/Kushalp2004/EnviroScan/blob/main/Enviroscan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install osmx

import pandas as pd
import requests
import osmx as ox

#Fetching AQI data from OpenAQ
def fetch_openaq_data(city, params):
  url = f"https://api.openaq.org/v2/measurements?city={city}"
  response = requests.get(url, params=params)
  data = response.json()['results']
  return pd.DataFrame(data)

#Dataset of the pollution
#Get weather data from open weather map
def fetch_weather_data(lat, lon, api_key):
  url = f"https://api.openweathermap.org/data/2.5/weather"
  params = {
      'lat': lat,
      'lon': lon,
      'appid': api_key
  }
  response = requests.get(url, params=params)
  return response.json()

#Getting location features from Open Street Map
def get_location_features(lat, lon, dist = 1000):
  G = ox.graph_from_point((lat, lon), dist = dist, network_type='Drive')
  roads = ox.geometries.geometries_from_point((lat, lon), tags={'highway': True}, dist = dist)
  factories = ox.geometries.geometries_from_point((lat, lon), tags={'landuse': 'industrial'}, dist = dist)
  return {'roads': roads, 'factories': factories}


In [21]:
# city = "Delhi"
# params = {
#     'parameter': 'pm25',
#     'limit': 1000
# }
# aqi_data = fetch_openaq_data(city, params)
# print(aqi_data)

In [22]:
def clean_pollution_data(df):
  df = df.drop_duplicates()
  df = df.dropna(subset=['value', 'coordinates.latitude', 'coordinates.longitude'])
  df['value'] = pd.to_numeric(df['value'])
  df['timestamp'] = pd.to_datetime(df['data']['utc'])
  #Input missing values
  df = df.fillna(df.mean())
  return df

def feature_engineering(df):
  #Normalizing polluant values
  for col in ['value']:
    df[col] = (df[col] - df[col].mean()) / df[col].std()

  #Adding temporal features
  df['hour'] = df['timestamp'].dt.hour
  df['day_of_week'] = df['timestamp'].dt.dayofweek

  return df


In [23]:
#Labelling of source
def label_source(df):
  df['source'] = 'Unknown'
  df.loc[(df['near_main_road'] == 1) & (df['NO2'] > 40), 'source'] = 'Vehicular'
  df.loc[(df['near_factory'] == 1) & (df['SO2'] <= 40), 'source'] = 'Industrial'
  df.loc[(df['near_farmland'] == 1) & (df['season'] == 'Dry') & (df['PM2.5'] > 70), 'source'] = 'Agricultural'

  return df


In [24]:
#Model training and source prediction

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report

def train_predict_model(df):
    features = ['PM2.5', 'NO2', 'SO2', 'CO', 'roads_proximity', 'factories_proximity', 'temperature', 'humidity', 'hour', 'dayofweek']
    X = df[features]
    y = df['source']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
    clf = RandomForestClassifier()
    param_grid = {'n_estimators': [50, 100], 'max_depth': [5, 10, None]}
    grid = GridSearchCV(clf, param_grid)
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)
    print(classification_report(y_test, y_pred))
    return grid.best_estimator_


In [25]:
import folium

def plot_heatmap(df):
    m = folium.Map(location=[df['latitude'].mean(), df['longitude'].mean()], zoom_start=12)
    for _, row in df.iterrows():
        folium.Circle(
            location=[row['latitude'], row['longitude']],
            radius=50,
            color="red" if row['source'] == "Industrial" else "blue",
            fill=True
        ).add_to(m)
    return m

# m = plot_heatmap(labeled_df)
# m.save('map.html')

In [19]:
!pip install streamlit
!pip install folium
!pip install pyngrok
#os.environ["NGROK_AUTHTOKEN"] = "32Hag184orm897nxmZXWOICPHhs_6CuuKPUsCYnYUzr9mqjRR"
#os.environ["NGROK_AUTHTOKEN"] = "32Q4NMHhzSBUduYMBv5Oyh54jg5_2toWjuF8icc3rJjCc9ZTr"
!pip install -q streamlit pyngrok

In [26]:
%%writefile app.py

import streamlit as st

st.set_page_config(page_title="Pollution Source Identification")
st.title("Ai Powered Pollution Source Identifier")

city = st.text_input("Enter the city name: ", placeholder="eg: Delhi")

if st.button("Analyze"):
  if city.strip() == "":
    st.warning("Please enter a valid city name. ")
  else:
    st.success("Analyzing pollution sources for {city}")
    st.markdown(f"""
          AI Powered Analysis for {city} (Simulated)
          Main Pollutants: PM2.5, NOx, SO2
          Likely Sources:
           - Industrial
           - Agricultural Burning/ Garbage burning
           - Vehicular
          Air Quality Index (AQI) : 185 (Unhealthy)
          Recommendation: Limit outdoor activities and stay indoors. Use Masks. Air purifiers are recommended for indoors
    """)

Overwriting app.py


In [27]:
# Install necessary modules
!pip install pyngrok streamlit --quiet

# Import necessary libraries
from pyngrok import ngrok
import time, os

# Put your ngrok token for authentication
os.environ["NGROK_AUTHTOKEN"] = "32Q4G4mxJ0vxU9gXE2PJ36tvknx_vkVF36vhUeWnXtxkodwN"

# Authenticate pyngrok using the exported token
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

# Kill any existing Streamlit process
!pkill streamlit

# Define your Streamlit app code
app_code = '''
import streamlit as st

st.set_page_config(page_title="Streamlit via ngrok", page_icon="🎈")
st.title("🚀 Hello from Streamlit!")
st.write("This Streamlit app is running through an ngrok tunnel.")
'''

# Write app code to a file
with open("app.py", "w") as f:
    f.write(app_code)

# Start the Streamlit app in the background
!streamlit run app.py &> /content/logs.txt &

# Wait a few seconds for the app to boot
time.sleep(15)

# Open an ngrok tunnel to port 8501
public_url = ngrok.connect(8501)

# Print the public URL
print("🌏 Your Streamlit app is live at:", public_url)


🌏 Your Streamlit app is live at: NgrokTunnel: "https://7755d98b3014.ngrok-free.app" -> "http://localhost:8501"


In [ ]:
# Install required packages (run once)
!pip install --quiet streamlit pyngrok

import os
import time
import subprocess
from pyngrok import ngrok

# Set your ngrok auth token (replace with your actual token)
NGROK_AUTH_TOKEN = "32Q4G4mxJ0vxU9gXE2PJ36tvknx_vkVF36vhUeWnXtxkodwN"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Write a simple Streamlit app
app_code = """
import streamlit as st

st.title("🚀 EnviroScan Pollution Source Identifier")

city = st.text_input("Enter a city name", "Delhi")
if st.button("Analyze"):
    if not city.strip():
        st.warning("Please enter a valid city name.")
    else:
        st.success(f"Analyzing pollution sources for: {city}")
        st.markdown('''
        ### AI Analysis Results for **{city}** (Simulated)
        - **Main Pollutants:** PM2.5, NOx, SO2
        - **Likely Sources:**
          - Vehicle emissions
          - Industrial activity
          - Biomass/garbage burning
        - **Air Quality Index (AQI):** 185 (Unhealthy)
        - **Recommendation:** Limit outdoor activity. Use masks. Air purifiers recommended indoors.
        '''.format(city=city))
"""

with open("app.py", "w") as f:
    f.write(app_code)

# Kill previous Streamlit instances
subprocess.run(["pkill", "-f", "streamlit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Disconnect any existing ngrok tunnels to avoid free-tier limits
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# Start Streamlit app in the background
streamlit_process = subprocess.Popen(["streamlit", "run", "app.py"])

# Wait for Streamlit server to start
time.sleep(15)  # Increase if needed

# Open ngrok tunnel to port 8501
public_url = ngrok.connect(8501)
print(f"🌐 Your Streamlit app is live at: {public_url}")

# Keep process alive to maintain server & tunnel
try:
    streamlit_process.wait()
except KeyboardInterrupt:
    streamlit_process.terminate()
    ngrok.disconnect(public_url)
    print("Terminated Streamlit and ngrok tunnel.")

🌐 Your Streamlit app is live at: NgrokTunnel: "https://5dbfb1bacaa0.ngrok-free.app" -> "http://localhost:8501"
